# Interfaces and Abstract Classes

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Write an interface as a contract and classes that implement it, and use them through the interface type.
- Write an abstract class that does the common work and leaves abstract methods for each subclass.
- Choose between an interface and an abstract class for a design.

**In this notebook**, one idea at a time (run each cell before moving on):
1. An interface is a contract
2. Several interfaces, and default methods
3. Abstract classes
4. Interface or abstract class?
5. A real interface: Comparable
6. Common mistakes

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. An interface is a contract

An **interface** lists methods without code: it says *what* a class can do, not *how*. A class promises to have those methods with `implements`, and must write every one of them, as `public`. Code can then use any of those classes through the interface type, without knowing which one it has.

In [ ]:
interface Discount {
    double apply(double price);
}

class Percent implements Discount {
    private final int percent;

    Percent(int percent) {
        this.percent = percent;
    }

    public double apply(double price) {
        return price * (100 - percent) / 100;
    }
}

class Fixed implements Discount {
    public double apply(double price) {
        return Math.max(0, price - 10);
    }
}

void main() {
    List<Discount> offers = List.of(new Percent(20), new Fixed());
    for (Discount d : offers) {
        IO.println(d.getClass().getSimpleName() + ": " + d.apply(30));
    }
}

🧪 **Try it:** remove `public` from one `apply` and read the error. Then write a third discount, `HalfPrice`, and add it to the list.

### ✏️ Exercise: Upper-case score

Finish `UpperScorer`: its score is the number of upper-case letters in the word. `best` already works with any Scorer, so it picks the word with most capitals.

In [ ]:
interface Scorer {
    int score(String word);
}

class LengthScorer implements Scorer {
    public int score(String word) {
        return word.length();
    }
}

class UpperScorer implements Scorer {
    public int score(String word) {
        // TODO: count the upper-case letters (Character.isUpperCase)
        return 0;
    }
}

String best(List<String> words, Scorer scorer) {
    String best = "";
    for (String w : words) {
        if (best.isEmpty() || scorer.score(w) > scorer.score(best)) {
            best = w;
        }
    }
    return best;
}

void main() {
    check("three capitals", new UpperScorer().score("HeLLo"), 3);
    check("no capitals", new UpperScorer().score("quiet"), 0);
    List<String> words = List.of("banana", "NASA", "JavaScript");
    check("by length", best(words, new LengthScorer()), "JavaScript");
    check("by capitals", best(words, new UpperScorer()), "NASA");
}

<details>
<summary>Show an answer</summary>

```java
interface Scorer {
    int score(String word);
}

class LengthScorer implements Scorer {
    public int score(String word) {
        return word.length();
    }
}

class UpperScorer implements Scorer {
    public int score(String word) {
        int count = 0;
        for (int i = 0; i < word.length(); i++) {
            if (Character.isUpperCase(word.charAt(i))) {
                count++;
            }
        }
        return count;
    }
}

String best(List<String> words, Scorer scorer) {
    String best = "";
    for (String w : words) {
        if (best.isEmpty() || scorer.score(w) > scorer.score(best)) {
            best = w;
        }
    }
    return best;
}

void main() {
    check("three capitals", new UpperScorer().score("HeLLo"), 3);
    check("no capitals", new UpperScorer().score("quiet"), 0);
    List<String> words = List.of("banana", "NASA", "JavaScript");
    check("by length", best(words, new LengthScorer()), "JavaScript");
    check("by capitals", best(words, new UpperScorer()), "NASA");
}
```

</details>

## 2. Several interfaces, and default methods

A class can extend only one class, but it can implement **many** interfaces, separated by commas. An interface can also hold a **`default` method**, with code, which every implementing class gets for free and can still override.

In [ ]:
interface Named {
    String name();

    default String greeting() {          // code every Named class gets
        return "Hi, I'm " + name();
    }
}

interface Priced {
    double price();
}

class Product implements Named, Priced {
    public String name() {
        return "Notebook";
    }

    public double price() {
        return 12.5;
    }
}

void main() {
    Product p = new Product();
    IO.println(p.greeting());
    Priced asPriced = p;                 // the same object, seen through another contract
    IO.println(asPriced.price());
}

🧪 **Try it:** override `greeting()` in Product. Then try `asPriced.name()`: through the `Priced` type, only `price()` is visible.

## 3. Abstract classes

An **abstract class** is a parent that can't be created with `new`. Unlike an interface, it can hold **fields, constructors and finished methods**, and it marks some methods `abstract`: no body, and every subclass must write them. A common design is a finished method that does the shared work and calls the abstract one for the part that changes. The slides follow it.

#### 🎞️ An abstract class with a part left open

![p.price() runs the price() written once in the abstract class Ticket, which calls the abstract discount(); StudentTicket supplies 0.5, so the price is 50.0; new Ticket(...) is refused because Ticket is abstract](template-method-steps.svg)

In [ ]:
interface Priced {
    double price();
}

abstract class Ticket implements Priced {
    final String event;

    Ticket(String event) {
        this.event = event;
    }

    abstract double discount();             // each kind of ticket decides

    public double price() {                 // the shared work, written once
        return 100 * (1 - discount());
    }
}

class StudentTicket extends Ticket {
    StudentTicket(String event) {
        super(event);
    }

    double discount() {
        return 0.5;
    }
}

class FullTicket extends Ticket {
    FullTicket(String event) {
        super(event);
    }

    double discount() {
        return 0;
    }
}

void main() {
    List<Ticket> sold = List.of(new StudentTicket("Show"), new FullTicket("Show"));
    for (Ticket t : sold) {
        IO.println(t.event + ": " + t.price());
    }
}

🧪 **Try it:** try `new Ticket("Show")` and read the error. Then remove `discount()` from FullTicket: the compiler requires every abstract method.

### ✏️ Exercise: Express shipping

`Shipping` charges 5 plus a price per kilo, and leaves `perKg()` abstract. Make `Express` charge 4.5 per kilo.

In [ ]:
abstract class Shipping {
    abstract double perKg();

    double cost(double kg) {
        return 5 + perKg() * kg;
    }
}

class Standard extends Shipping {
    double perKg() {
        return 2;
    }
}

class Express extends Shipping {
    double perKg() {
        // TODO: 4.5 per kilo
        return 0;
    }
}

void main() {
    check("standard, 2 kg", new Standard().cost(2), 9.0);
    check("express, 2 kg", new Express().cost(2), 14.0);
    Shipping s = new Express();
    check("express through Shipping, 0 kg", s.cost(0), 5.0);
}

<details>
<summary>Show an answer</summary>

```java
abstract class Shipping {
    abstract double perKg();

    double cost(double kg) {
        return 5 + perKg() * kg;
    }
}

class Standard extends Shipping {
    double perKg() {
        return 2;
    }
}

class Express extends Shipping {
    double perKg() {
        return 4.5;
    }
}

void main() {
    check("standard, 2 kg", new Standard().cost(2), 9.0);
    check("express, 2 kg", new Express().cost(2), 14.0);
    Shipping s = new Express();
    check("express through Shipping, 0 kg", s.cost(0), 5.0);
}
```

</details>

## 4. Interface or abstract class?

| | Interface | Abstract class |
|---|---|---|
| A class can have… | many (`implements A, B`) | only one (`extends`) |
| Fields | no (only constants) | yes |
| Constructor | no | yes |
| Methods with code | `default` methods | any |
| Fits | a **capability** shared by unrelated classes: `Comparable`, `Priced` | a **family** of related classes sharing state and code: `Ticket` |

A common mix: an interface for the contract, and an abstract class that implements it with the shared code, as `Ticket implements Priced` above.

## 5. A real interface: Comparable

Java's own library is full of interfaces. `Comparable<T>` has one method, `compareTo`, which returns a negative number, zero or a positive number. A class that implements it gets a natural order, so `Collections.sort` and `TreeSet` can sort its objects.

In [ ]:
class Player implements Comparable<Player> {
    final String name;
    final int score;

    Player(String name, int score) {
        this.name = name;
        this.score = score;
    }

    public int compareTo(Player other) {
        return Integer.compare(other.score, score);   // the highest score first
    }

    @Override
    public String toString() {
        return name + " " + score;
    }
}

void main() {
    List<Player> table = new ArrayList<>(List.of(new Player("Ana", 30), new Player("Bia", 75), new Player("Caio", 50)));
    Collections.sort(table);
    IO.println(table);
}

🧪 **Try it:** swap the arguments of `Integer.compare` to sort from the lowest score. Then put the players in a `TreeSet` instead of sorting the list.

## 6. Common mistakes

- **`new` on an interface or an abstract class:** neither can be created on its own; create a class that implements or extends it.
- **A missing method:** a class that implements an interface or extends an abstract class must write every abstract method, or be abstract itself.
- **Forgetting `public`** on a method that implements an interface method.
- **`extends` with an interface, or `implements` with a class:** classes are extended; interfaces are implemented.

In [ ]:
interface Shape {
    double area();
}

class Square implements Shape {
    private final double side;

    Square(double side) {
        this.side = side;
    }

    public double area() {
        return side * side;
    }
}

void main() {
    Shape s = new Square(3);
    IO.println(s.area());
}

🧪 **Try it:** try `new Shape()`, then write `class Square extends Shape` and read each error.

## In short

An **interface** is a contract: it lists methods without code, and any class can promise to have them with `implements`. An **abstract class** is a parent that can't be created with `new`: it holds fields and finished methods, and leaves some methods `abstract` for each subclass to write.

```java
interface Priced {
    double price();
}

abstract class Ticket implements Priced {
    final String event;

Ticket(String event) {
        this.event = event;
    }

abstract double discount();

public double price() {
        return 100 * (1 - discount());
    }
}

class StudentTicket extends Ticket {
    StudentTicket(String event) {
        super(event);
    }

double discount() {
        return 0.5;
    }
}

void main() {
    Priced p = new StudentTicket("Concert");
    IO.println(p.price());
}
```

Use an **interface** when unrelated classes share a capability (a class can implement many). Use an **abstract class** when related classes share state and code (a class extends only one).

**Watch out:** methods that implement an interface must be `public`, because interface methods are public.

## Readings

- [Abstract Methods and Classes](https://dev.java/learn/language/oop/inheritance/abstract-classes/) (dev.java (Oracle), 7 min): What makes a method or a class abstract, then the section "Abstract Classes Compared to Interfaces", which says when to choose each one, and the GraphicObject example.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Animals (Interfaces)** · Medium
- **Notifications (Abstract Class)** · Medium